# Phase 3 — Ultra-short fixed context with LSTM

Here **few-shot** means a **fixed tiny history** (e.g. 5–8 frames): `GTSequenceDataset` with small `seq_in_len` and matching `seq_total_len = seq_in_len + seq_out_len`. No padding mask is required; the recurrent model only sees real frames.

Compare validation loss to long-context training (`phase1_improved.ipynb`): short context is harder but closer to cold-start tracking after a short tracklet.

In [ ]:
import os
import torch
from torch import optim
from torch.utils.data import DataLoader

from dataset import GTSequenceDataset
from lstm_improved import ImprovedLSTMPredictor
from loss import LossFunction

SEQ_IN_LEN = 6
SEQ_OUT_LEN = 20
SEQ_TOTAL_LEN = SEQ_IN_LEN + SEQ_OUT_LEN
BATCH_SIZE = 512
STEPS = 4
NOISE_COEFF = 0.12
NOISE_PROB = 0.15
BASE_DIR = os.environ.get("MOT_DATASET_ROOT", "../../Datasets/")

train_ds = GTSequenceDataset.from_roots(
    [f"{BASE_DIR}MOT17/train"],
    seq_in_len=SEQ_IN_LEN,
    seq_out_len=SEQ_OUT_LEN,
    seq_total_len=SEQ_TOTAL_LEN,
    steps=STEPS,
    noise_prob=NOISE_PROB,
    noise_coeff=NOISE_COEFF,
)
val_ds = GTSequenceDataset.from_roots(
    [f"{BASE_DIR}MOT17/val"],
    seq_in_len=SEQ_IN_LEN,
    seq_out_len=SEQ_OUT_LEN,
    seq_total_len=SEQ_TOTAL_LEN,
    steps=STEPS,
    noise_prob=NOISE_PROB,
    noise_coeff=NOISE_COEFF,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print("train", len(train_ds), "val", len(val_ds))

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = ImprovedLSTMPredictor(
    input_dim=13,
    d_model=256,
    hidden_dim=256,
    num_layers=2,
    dropout=0.1,
    teacher_forcing_ratio=0.45,
).to(DEVICE)
crit = LossFunction(loss1_coeff=1.0, loss2_coeff=0.4, loss3_coeff=0.25, loss4_coeff=0.0)
opt = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=25)

for ep in range(1, 26):
    tr = model.train_one_epoch(train_loader, opt, crit, device=DEVICE)
    va = model.evaluate(val_loader, crit, device=DEVICE)
    sched.step()
    print(f"ep {ep:02d} train {tr:.5f} val {va:.5f}")

os.makedirs("pretrained", exist_ok=True)
model.save_weight("pretrained/lstm_few_shot_ctx6.pth")
print("saved pretrained/lstm_few_shot_ctx6.pth")